### Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Runtime` -> `Change runtime type` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [3]:
!nvidia-smi

Wed Jun  3 01:23:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [4]:
import os
HOME = os.getcwd()
print(HOME)

/content


### Install dependencies required for YOLO26

In [5]:
%pip install -q "ultralytics>=8.4.0" supervision roboflow

# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 51.2/112.6 GB disk)


## Fine-tune YOLO26 on custom dataset

**NOTE:** When training YOLO26, make sure your data is located in `datasets`. If you'd like to change the default location of the data you want to use for fine-tuning, you can do so through Ultralytics' `settings.json`. In this tutorial, we will use one of the [datasets](https://universe.roboflow.com/liangdianzhong/-qvdww) available on [Roboflow Universe](https://universe.roboflow.com/). When downloading, make sure to select the `yolov11` export format.

In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="x3bqAMMnDKugRpXQ1kjz")
project = rf.workspace("homeworkspace").project("bd-license-plat-detection")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to BD-License-Plat-Detection-1 in yolo26:: 100%|██████████| 2671/2671 [00:00<00:00, 8109.76it/s]


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Custom Training

In [8]:
# 1. Mount your Google Drive
from google.colab import drive
import shutil
import os

# This will trigger a popup asking you to authorize Colab to access your Drive
drive.mount('/content/drive')

# 2. Install Ultralytics
!pip install ultralytics
import ultralytics
ultralytics.checks()
from ultralytics import YOLO

# 3. Load and Train the model
model = YOLO("yolo26n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    patience=10,
    imgsz=640,
    batch=16,
    device=0,
    name="bd_plates_yolo26"
)

# 4. Automatically copy the best model to your Google Drive!
source_path = '/content/runs/detect/bd_plates_yolo26/weights/best.pt'
destination_path = '/content/drive/MyDrive/bd_plates_best_yolo26.pt'

if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print(f"\n✅ SUCCESS: Your model is safely backed up to your Google Drive at: {destination_path}")
else:
    print("\n❌ ERROR: Could not find the best.pt file. Check the training logs above.")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.4/112.6 GB disk)
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/BD-License-Plat-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.